# Trabalho Prático 05 — Othello com Agentes Inteligentes

**Disciplina:** GCC128 — Inteligência Artificial  
**Tecnologias:** Python 3, Pygame, LangChain, LangGraph e Groq  

Objetivo: desenvolver um agente que observa o estado do Othello, consulta ferramentas, analisa jogadas, decide e executa ações. O projeto também permitirá comparar estratégias.

> **Tarefa 2:** configuração inicial. As regras do Othello serão implementadas na Tarefa 3.

## 1. Dependências

Execute a célula abaixo **uma vez** no kernel escolhido. O comando `%pip` instala as bibliotecas no ambiente Python do próprio notebook. Recomenda-se utilizar um kernel associado a um ambiente virtual local.

In [8]:
%pip install -q pygame langchain langgraph langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Verificação do ambiente

Confere se as bibliotecas foram instaladas e podem ser importadas. Não acessa a API e não abre uma janela gráfica.

In [9]:
import sys
from importlib.metadata import version

import pygame
import langchain
import langgraph
import langchain_groq

print("Python:", sys.version.split()[0])
for pacote in ("pygame", "langchain", "langgraph", "langchain-groq"):
    print(f"{pacote}: {version(pacote)}")
print("Ambiente configurado. Nenhum agente foi executado.")

Python: 3.13.2
pygame: 2.6.1
langchain: 1.4.9
langgraph: 1.2.20
langchain-groq: 1.1.3
Ambiente configurado. Nenhum agente foi executado.


## 3. Próxima etapa — Motor do Othello

Reservado para a **Tarefa 3**: tabuleiro 8×8, jogadas válidas, captura de peças, alternância de turnos e fim de partida.


3.1 Inicialização do tabuleiro

In [ ]:
# Representação das casas
VAZIO = 0
PRETO = 1
BRANCO = -1

TAMANHO = 8

def criar_tabuleiro():
    tabuleiro = [
        [VAZIO for _ in range(TAMANHO)]
        for _ in range(TAMANHO)
    ]

    # Posições iniciais das peças
    tabuleiro[3][3] = BRANCO
    tabuleiro[3][4] = PRETO
    tabuleiro[4][3] = PRETO
    tabuleiro[4][4] = BRANCO

    return tabuleiro


def exibir_tabuleiro(tabuleiro):
    simbolos = {
        VAZIO: ".",
        PRETO: "P",
        BRANCO: "B"
    }

    print("    " + " ".join(str(i) for i in range(TAMANHO)))

    for i, linha in enumerate(tabuleiro):
        print(f"{i}   " + " ".join(simbolos[peca] for peca in linha))

    0 1 2 3 4 5 6 7
0   . . . . . . . .
1   . . . . . . . .
2   . . . . . . . .
3   . . . B P . . .
4   . . . P B . . .
5   . . . . . . . .
6   . . . . . . . .
7   . . . . . . . .


3.2 Verificação de jogadas válidas

In [11]:
# Oito direções possíveis: horizontal, vertical e diagonais
DIRECOES = [
    (-1, -1), (-1, 0), (-1, 1),
    (0, -1),           (0, 1),
    (1, -1),  (1, 0),  (1, 1)
]


def pecas_capturadas(tabuleiro, linha, coluna, jogador):
    if not (0 <= linha < TAMANHO and 0 <= coluna < TAMANHO):
        return []

    if tabuleiro[linha][coluna] != VAZIO:
        return []

    capturadas = []

    for dl, dc in DIRECOES:
        l, c = linha + dl, coluna + dc
        caminho = []

        # Percorre as peças adversárias nesta direção
        while (
            0 <= l < TAMANHO
            and 0 <= c < TAMANHO
            and tabuleiro[l][c] == -jogador
        ):
            caminho.append((l, c))
            l += dl
            c += dc

        # A sequência só é válida se terminar em uma peça própria
        if (
            caminho
            and 0 <= l < TAMANHO
            and 0 <= c < TAMANHO
            and tabuleiro[l][c] == jogador
        ):
            capturadas.extend(caminho)

    return capturadas


def movimentos_validos(tabuleiro, jogador):
    movimentos = []

    for linha in range(TAMANHO):
        for coluna in range(TAMANHO):
            if pecas_capturadas(tabuleiro, linha, coluna, jogador):
                movimentos.append((linha, coluna))

    return movimentos

3.3 executar jogadas e inverter peças

In [12]:
def fazer_jogada(tabuleiro, linha, coluna, jogador):
    # Identifica as peças que serão capturadas
    capturadas = pecas_capturadas(
        tabuleiro, linha, coluna, jogador
    )

    # Impede movimentos inválidos
    if not capturadas:
        return False

    # Coloca a peça do jogador
    tabuleiro[linha][coluna] = jogador

    # Inverte as peças adversárias capturadas
    for l, c in capturadas:
        tabuleiro[l][c] = jogador

    return True

3.4 Controle de turnos e passagem de ver


In [14]:
def proximo_turno(tabuleiro, jogador_atual):
    adversario = -jogador_atual

    # Verifica se o adversário pode jogar
    if movimentos_validos(tabuleiro, adversario):
        return adversario

    # Se o adversário não puder, o jogador atual continua
    if movimentos_validos(tabuleiro, jogador_atual):
        return jogador_atual

    # Nenhum jogador possui movimentos válidos
    return None

3.5 Encerramento do jogo e contagem de peças

In [16]:
def contar_pecas(tabuleiro):
    pretas = sum(linha.count(PRETO) for linha in tabuleiro)
    brancas = sum(linha.count(BRANCO) for linha in tabuleiro)

    return pretas, brancas


def jogo_terminou(tabuleiro):
    return (
        not movimentos_validos(tabuleiro, PRETO)
        and not movimentos_validos(tabuleiro, BRANCO)
    )


def obter_vencedor(tabuleiro):
    if not jogo_terminou(tabuleiro):
        return None

    pretas, brancas = contar_pecas(tabuleiro)

    if pretas > brancas:
        return PRETO
    elif brancas > pretas:
        return BRANCO
    else:
        return VAZIO  # Empate

### 4 - Implementação das Tools

In [21]:
def iniciar_partida():
    return {
        "tabuleiro": criar_tabuleiro(),
        "jogador_atual": PRETO,
        "historico": []
    }


# Estado compartilhado entre as ferramentas
estado_jogo = iniciar_partida()

4.2 - Ferramentas de consulta do jogo

In [23]:
from langchain_core.tools import tool


@tool
def get_board_state() -> dict:
    """Consulta o estado atual do tabuleiro de Othello."""

    tabuleiro = estado_jogo["tabuleiro"]
    pretas, brancas = contar_pecas(tabuleiro)

    return {
        "board": [linha.copy() for linha in tabuleiro],
        "current_player": estado_jogo["jogador_atual"],
        "black_count": pretas,
        "white_count": brancas
    }


@tool
def get_valid_moves() -> dict:
    """Consulta os movimentos válidos do jogador atual.
    As coordenadas de linha e coluna começam em zero.
    """

    jogador = estado_jogo["jogador_atual"]

    if jogador is None:
        return {"current_player": None, "moves": []}

    movimentos = movimentos_validos(
        estado_jogo["tabuleiro"], jogador
    )

    return {
        "current_player": jogador,
        "moves": [
            {"row": linha, "col": coluna}
            for linha, coluna in movimentos
        ]
    }

4.3 - Implmentação da ferramenta make_move()

In [25]:
@tool
def make_move(row: int, col: int) -> dict:
    """Executa uma jogada de Othello para o jogador atual.
    As coordenadas row e col devem estar entre 0 e 7.
    """

    tabuleiro = estado_jogo["tabuleiro"]
    jogador = estado_jogo["jogador_atual"]

    # Verifica se a partida já terminou
    if jogador is None:
        return {
            "success": False,
            "message": "A partida já terminou."
        }

    # Executa a jogada
    if not fazer_jogada(tabuleiro, row, col, jogador):
        return {
            "success": False,
            "message": "Jogada inválida.",
            "valid_moves": movimentos_validos(tabuleiro, jogador)
        }

    # Registra a jogada no histórico
    estado_jogo["historico"].append({
        "player": jogador,
        "row": row,
        "col": col
    })

    # Atualiza o jogador atual
    proximo = proximo_turno(tabuleiro, jogador)
    estado_jogo["jogador_atual"] = proximo

    return {
        "success": True,
        "message": "Jogada realizada com sucesso.",
        "next_player": proximo,
        "game_over": proximo is None
    }

4.4 - Implementação da ferramenta get_game_status()

In [27]:
@tool
def get_game_status() -> dict:
    """Consulta o estado da partida de Othello, incluindo
    pontuação, jogador atual, encerramento e vencedor.
    """

    tabuleiro = estado_jogo["tabuleiro"]

    pretas, brancas = contar_pecas(tabuleiro)
    terminou = jogo_terminou(tabuleiro)

    return {
        "game_over": terminou,
        "current_player": estado_jogo["jogador_atual"],
        "black_count": pretas,
        "white_count": brancas,
        "total_moves": len(estado_jogo["historico"]),
        "winner": obter_vencedor(tabuleiro)
    }